# 03 - Manufacturing Process Parameter Optimization

This notebook shows how Gaussian Process-based Bayesian Optimization can be used for manufacturing/process parameter tuning.

Scenario: a CNC operation has three controllable parameters:

- spindle speed,
- feed rate,
- depth of cut.

In a real factory, the objective could come from a physical trial, sensor system, quality laboratory, digital twin, or another expensive engineering analysis.

The objective used here is **synthetic and educational**. It is not a validated machining model or physical metal-cutting law.

## 1. Why Bayesian Optimization?

A dense factorial design grows quickly. If each of three parameters has 20 levels,

\[20^3 = 8000\]

experiments would be required.

If a physical test takes 15 minutes, a dense search can become impractical. Bayesian Optimization attempts to use a small initial design and then direct subsequent trials toward regions that are promising or informative according to the GP surrogate.

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import matplotlib.pyplot as plt
from sklearn.exceptions import ConvergenceWarning

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(
        "Run this notebook from the repository root or from the notebooks directory."
    )

sys.path.insert(0, str((PROJECT_ROOT / "src").resolve()))
from gaussian_bo import GaussianProcessBayesOptimizer

warnings.filterwarnings("ignore", category=ConvergenceWarning)

## 2. Decision variables

| Parameter | Lower bound | Upper bound |
|---|---:|---:|
| Spindle speed | 1500 rpm | 4500 rpm |
| Feed rate | 100 mm/min | 500 mm/min |
| Depth of cut | 0.5 mm | 3.0 mm |

The variables have very different physical scales. The optimizer internally maps all decision variables to `[0, 1]` before fitting the GP.

## 3. Synthetic black-box objective

The teaching objective combines several artificial effects:

- distance from a preferred operating region,
- parameter interactions,
- a productivity-like penalty,
- an energy-like penalty,
- a mild nonconvex component.

The coefficients are not manufacturing data.

In a real project, this function would be replaced by the actual experiment or simulation interface.

In [ ]:
def synthetic_cnc_objective(x):
    spindle_speed, feed_rate, depth_of_cut = map(float, x)

    # Normalize variables for the synthetic teaching formula.
    u = (spindle_speed - 1500.0) / (4500.0 - 1500.0)
    v = (feed_rate - 100.0) / (500.0 - 100.0)
    w = (depth_of_cut - 0.5) / (3.0 - 0.5)

    quality_loss = (
        85.0 * (u - 0.62) ** 2
        + 110.0 * (v - 0.43) ** 2
        + 70.0 * (w - 0.55) ** 2
    )

    interaction_loss = 22.0 * (u - v) ** 2 + 10.0 * (v - w) ** 2
    productivity_loss = 18.0 / (0.25 + v) + 8.0 / (0.25 + w)
    energy_like_loss = 8.0 * u**2 + 5.0 * w**2
    nonconvex_component = 2.5 * np.sin(6.0 * u) * np.cos(4.0 * v)

    return (
        quality_loss
        + interaction_loss
        + productivity_loss
        + energy_like_loss
        + nonconvex_component
    )

## 4. Run Bayesian Optimization

The evaluation budget is

```text
8 initial evaluations + 25 sequential evaluations = 33 evaluations
```

This is much smaller than the illustrative 8000-point factorial grid. However, the resulting solution is the **best found under the budget**, not a guaranteed global optimum.

In [ ]:
optimizer = GaussianProcessBayesOptimizer(
    objective_function=synthetic_cnc_objective,
    bounds=[
        [1500.0, 4500.0],
        [100.0, 500.0],
        [0.5, 3.0],
    ],
    n_initial_points=8,
    acquisition_function="ei",
    xi=0.01,
    random_state=42,
)

result = optimizer.optimize(n_iterations=25, verbose=True)
spindle_speed, feed_rate, depth_of_cut = result.best_x

print("Best parameters found:")
print(f"Spindle speed: {spindle_speed:.1f} rpm")
print(f"Feed rate: {feed_rate:.1f} mm/min")
print(f"Depth of cut: {depth_of_cut:.3f} mm")
print(f"Synthetic objective value: {result.best_y:.4f}")

## 5. Convergence

In [ ]:
best_so_far = np.minimum.accumulate(result.y_observed)

plt.figure(figsize=(9, 5))
plt.plot(np.arange(1, len(best_so_far) + 1), best_so_far, marker="o")
plt.xlabel("Experiment / objective evaluation count")
plt.ylabel("Best synthetic loss observed so far")
plt.title("Manufacturing process-parameter optimization")
plt.grid(True)
plt.show()

## 6. Moving from a synthetic example to a real process

Several issues become critical in a real manufacturing study.

### Replication and measurement noise

The same parameter combination may need multiple parts or repeated runs so that mean response and variance can be estimated.

### Constraints

Examples include tool temperature, surface-roughness limits, machine-power limits, cutting-force limits, or safety boundaries. Constraint treatment should be explicit rather than hidden inside arbitrary penalties whenever possible.

### Multiple objectives

Manufacturing often involves competing objectives such as surface finish, cycle time, energy, tool life, and scrap. Options include economic conversion to a common cost, weighted scalarization, constrained optimization, or multi-objective Bayesian Optimization.

## 7. Relationship to Design of Experiments

Bayesian Optimization does not replace DOE in every setting.

Classical DOE is especially valuable when the goal is to estimate factor effects, analyze interactions statistically, or support inferential conclusions. Bayesian Optimization is mainly oriented toward **sequentially locating high-performing parameter regions with a limited expensive experimental budget**.

The two approaches can be combined: use DOE or Latin Hypercube Sampling for the initial design and continue with sequential Bayesian Optimization.

## 8. Reporting checklist for an industrial project

A real project report should state at least:

1. decision variables and physical bounds,
2. units of the objective,
3. constraints,
4. initial experimental design,
5. GP kernel choice,
6. noise model,
7. acquisition function,
8. total experimental budget,
9. best parameters found,
10. independent validation runs,
11. baseline-method comparison,
12. an explicit statement that a global optimum is not guaranteed unless separately verified.